## Código Prerrequisito

Cargamos el código de las secciones anteriores sobre Datasets & DataLoaders y Construcción de Modelos.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
# Cargar datos de entrenamiento
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

# Cargar datos de prueba
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

# Crear DataLoaders
train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

In [ ]:
# Definir la arquitectura de la red neuronal
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

## Hiperparámetros

Los hiperparámetros son parámetros ajustables que te permiten controlar el proceso de optimización del modelo. Diferentes valores de hiperparámetros pueden impactar el entrenamiento del modelo y las tasas de convergencia.

Definimos los siguientes hiperparámetros para el entrenamiento:

- **Número de Épocas (*Epochs*)**: El número de veces que se itera sobre todo el conjunto de datos. Una época completa significa que el modelo ha visto todos los ejemplos de entrenamiento una vez.

- **Tamaño de Lote (*Batch Size*)**: El número de muestras de datos que se propagan a través de la red antes de que los parámetros se actualicen. Lotes más grandes requieren más memoria pero pueden ser más eficientes computacionalmente.

- **Tasa de Aprendizaje (*Learning Rate*)**: Cuánto actualizar los parámetros del modelo en cada lote/época. Valores pequeños producen una velocidad de aprendizaje lenta, mientras que valores grandes pueden resultar en un comportamiento impredecible durante el entrenamiento.

### ¿Cómo elegir los hiperparámetros?

- **Learning Rate**: Típicamente entre 0.001 y 0.1. Si la pérdida no disminuye, prueba con un valor más pequeño.
- **Batch Size**: Potencias de 2 (32, 64, 128, 256) para eficiencia computacional.
- **Epochs**: Aumenta hasta que el modelo deje de mejorar en el conjunto de validación.

In [ ]:
learning_rate = 1e-3  # 0.001
batch_size = 64
epochs = 5

## Bucle de Optimización

Una vez que establecemos nuestros hiperparámetros, podemos entrenar y optimizar nuestro modelo con un bucle de optimización. Cada iteración del bucle de optimización se llama una **época**.

Cada época consiste en dos partes principales:

1. **Bucle de Entrenamiento (*Train Loop*)**: Itera sobre el conjunto de datos de entrenamiento e intenta converger a parámetros óptimos.

2. **Bucle de Validación/Prueba (*Validation/Test Loop*)**: Itera sobre el conjunto de datos de prueba para verificar si el rendimiento del modelo está mejorando.

### Proceso en cada iteración del bucle de entrenamiento:

1. **Forward Pass**: Los datos pasan por el modelo para obtener predicciones
2. **Cálculo de Pérdida**: Se mide qué tan incorrectas son las predicciones
3. **Backward Pass**: Se calculan los gradientes de la pérdida
4. **Actualización de Parámetros**: Se ajustan los pesos usando los gradientes

## Función de Pérdida

Cuando se presenta con algunos datos de entrenamiento, nuestra red no entrenada probablemente no dará la respuesta correcta. La **función de pérdida** mide el grado de discrepancia entre el resultado obtenido y el valor objetivo, y es la función de pérdida la que queremos minimizar durante el entrenamiento.

Para calcular la pérdida, hacemos una predicción usando las entradas de nuestra muestra de datos dada y la comparamos con el valor real de la etiqueta.

### Funciones de Pérdida Comunes:

- **`nn.MSELoss`** (*Mean Square Error*): Para tareas de regresión
  - Fórmula: $MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$
  - Penaliza más los errores grandes

- **`nn.NLLLoss`** (*Negative Log Likelihood*): Para clasificación con log-probabilidades
  - Fórmula: $NLL = -\sum_{i=1}^{n} y_i \log(\hat{y}_i)$

- **`nn.CrossEntropyLoss`**: Combina `nn.LogSoftmax` y `nn.NLLLoss`
  - Ideal para clasificación multiclase
  - Aplica softmax automáticamente a los logits
  - Fórmula: $CE = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$

Pasamos los logits de salida de nuestro modelo a `nn.CrossEntropyLoss`, que normalizará los logits y calculará el error de predicción.

In [ ]:
# Inicializar la función de pérdida
loss_fn = nn.CrossEntropyLoss()

## Optimizador

La optimización es el proceso de ajustar los parámetros del modelo para reducir el error del modelo en cada paso de entrenamiento. Los **algoritmos de optimización** definen cómo se realiza este proceso.

Toda la lógica de optimización está encapsulada en el objeto `optimizer`. Aquí usamos el optimizador SGD (*Stochastic Gradient Descent*); además, hay muchos optimizadores diferentes disponibles en PyTorch que funcionan mejor para diferentes tipos de modelos y datos.

### Optimizadores Comunes:

- **SGD** (*Stochastic Gradient Descent*): Actualización básica con gradientes
  - Parámetros: `lr` (learning rate), `momentum` (opcional), `weight_decay` (regularización)
  - Fórmula: $\theta = \theta - \eta \cdot \nabla L(\theta)$

- **Adam**: Combina momentum y tasas de aprendizaje adaptativas
  - Muy popular y efectivo en la mayoría de casos
  - Parámetros: `lr`, `betas` (factores de decaimiento), `eps` (estabilidad numérica)

- **RMSProp**: Tasas de aprendizaje adaptativas
  - Bueno para problemas con gradientes no estacionarios

- **AdamW**: Variante de Adam con mejor regularización
  - Recomendado para transformers y modelos modernos

Inicializamos el optimizador registrando los parámetros del modelo que necesitan ser entrenados, y pasando el hiperparámetro de tasa de aprendizaje.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Pasos de Optimización

Dentro del bucle de entrenamiento, la optimización ocurre en tres pasos:

1. **`optimizer.zero_grad()`**: Reinicia los gradientes de los parámetros del modelo. Los gradientes por defecto se acumulan; para evitar el doble conteo, los ponemos explícitamente a cero en cada iteración.

2. **`loss.backward()`**: Retropropagación de la pérdida de predicción. PyTorch deposita los gradientes de la pérdida con respecto a cada parámetro.

3. **`optimizer.step()`**: Una vez que tenemos nuestros gradientes, llamamos a `optimizer.step()` para ajustar los parámetros según los gradientes recolectados en el paso hacia atrás (*backward pass*).

### ¿Por qué en este orden?

El orden es crucial:
- Primero limpiamos gradientes antiguos
- Luego calculamos nuevos gradientes con `backward()`
- Finalmente actualizamos parámetros con `step()`

Algunos prefieren llamar `zero_grad()` después de `step()`, ambos funcionan pero llamarlo al inicio es más claro conceptualmente.

## Implementación Completa

Definimos `train_loop` que itera sobre nuestro código de optimización, y `test_loop` que evalúa el rendimiento del modelo contra nuestros datos de prueba.

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    """
    Bucle de entrenamiento que optimiza los parámetros del modelo.
    
    Args:
        dataloader: DataLoader con los datos de entrenamiento
        model: Modelo a entrenar
        loss_fn: Función de pérdida
        optimizer: Optimizador para actualizar parámetros
    """
    size = len(dataloader.dataset)
    # Configurar el modelo en modo entrenamiento - importante para batch normalization y dropout
    # No es necesario en esta situación pero se añade como buena práctica
    model.train()
    
    for batch, (X, y) in enumerate(dataloader):
        # Calcular predicción y pérdida
        pred = model(X)
        loss = loss_fn(pred, y)

        # Retropropagación
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test_loop(dataloader, model, loss_fn):
    """
    Bucle de evaluación que mide el rendimiento del modelo.
    
    Args:
        dataloader: DataLoader con los datos de prueba
        model: Modelo a evaluar
        loss_fn: Función de pérdida
    """
    # Configurar el modelo en modo evaluación - importante para batch normalization y dropout
    # No es necesario en esta situación pero se añade como buena práctica
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluar el modelo con torch.no_grad() asegura que no se calculen gradientes durante el modo de prueba
    # también sirve para reducir cálculos de gradientes innecesarios y uso de memoria para tensores con requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## Entrenamiento del Modelo

Inicializamos la función de pérdida y el optimizador, y los pasamos a `train_loop` y `test_loop`. Puedes aumentar el número de épocas para rastrear el rendimiento mejorado del modelo.

### Interpretación de Resultados:

- **Loss (Pérdida)**: Debe disminuir con cada época. Si aumenta, algo está mal (learning rate muy alto, bug en el código).
- **Accuracy (Precisión)**: Debe aumentar con el tiempo. Si se estanca, el modelo puede haber alcanzado su capacidad o necesita más datos.
- **Overfitting**: Si la precisión de entrenamiento es mucho mayor que la de prueba, el modelo está sobreajustando.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

## Mejores Prácticas

### 1. Monitoreo durante el Entrenamiento
- Observa tanto la pérdida de entrenamiento como la de validación
- Guarda el modelo con mejor rendimiento en validación
- Usa TensorBoard o wandb para visualizar el progreso

### 2. Evitar Sobreajuste
- Usa **Dropout**: `nn.Dropout(p=0.5)` entre capas
- Aplica **Regularización**: `weight_decay` en el optimizador
- **Early Stopping**: Detén el entrenamiento cuando la validación deje de mejorar
- **Data Augmentation**: Aumenta la variedad de datos de entrenamiento

### 3. Optimización del Learning Rate
- Usa **Learning Rate Scheduler**: `torch.optim.lr_scheduler.StepLR`
- Reduce el learning rate cuando el entrenamiento se estanque
- Experimenta con valores: Si el loss explota → muy alto; Si no aprende → muy bajo

### 4. Batch Size
- Lotes más grandes: entrenamiento más estable pero necesitan más memoria
- Lotes más pequeños: más ruido pero a veces generaliza mejor
- Ajusta según tu hardware (GPU memory)

### 5. Guardar y Cargar Modelos
```python
# Guardar
torch.save(model.state_dict(), 'model.pth')

# Cargar
model = NeuralNetwork()
model.load_state_dict(torch.load('model.pth'))
model.eval()
```

## Resumen

En este notebook aprendimos:

1. **Hiperparámetros**: Learning rate, batch size y epochs controlan el proceso de entrenamiento
2. **Función de Pérdida**: Mide qué tan incorrectas son las predicciones del modelo
3. **Optimizador**: Ajusta los parámetros del modelo usando los gradientes calculados
4. **Bucle de Entrenamiento**: Forward pass → calcular pérdida → backward pass → actualizar parámetros
5. **Bucle de Evaluación**: Mide el rendimiento del modelo sin calcular gradientes
6. **Mejores Prácticas**: Monitoreo, regularización, ajuste de hiperparámetros

El proceso de entrenamiento es iterativo y experimental. No existe una configuración perfecta que funcione para todos los problemas, así que experimenta con diferentes hiperparámetros y arquitecturas.